# Trajectory Progression

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Score age-stratified ordering (Figure 6C)

In [ ]:
# Zoetrope vs STORIES: GW20+GW22 pooled, AGE-ADJUSTED IPC -> EN-Mig -> EN-ET + AUC.
# Uses precomputed, frozen Part-A matched-cell scores. NO retraining.
# Age adjustment for display:
#   (1) convert each method's scores to percentile ranks WITHIN each age;
#   (2) within each H1 group, give GW20 and GW22 EQUAL total weight.
# Means, quartiles and whiskers use the same age-balanced distribution.
# Colored diamonds and numerical annotations indicate AGE-ADJUSTED MEANS.
# AUC from original score values, within-age and H1-pair, then macro-average
# equally across 3 predeclared H1 pairs and both ages. No GW15: rho=0 there.

from pathlib import Path
import os, json, hashlib
import numpy as np
import pandas as pd
from scipy.stats import rankdata
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ---------------- Frozen matched-cell benchmark ----------------
PROJECT = Path(os.environ.get('SPATIAL_TI_PROJECT',
    os.environ["STP_PROJECT"]))
ROOT = Path(os.environ.get('QIAN_ORDERING_BENCH_OUT',
    str(PROJECT / 'paper_figures/qian_PFC_Zoetrope_vs_STORIES_ordering_v1')))
CANDIDATES = [
    ROOT / 'PART_A_POTENTIAL_ONLY_DEFAULT_ENV' / 'MATCHED_CELLS_STORIES_NATIVE_POTENTIAL.csv',
    ROOT / 'PART_A_MATCHED_CELLS_NATIVE_SCORES.csv',
]
SOURCE = next((p for p in CANDIDATES if p.is_file()), None)
if SOURCE is None:
    raise FileNotFoundError('Missing matched-cell Part-A benchmark CSV:\n' +
                            '\n'.join(map(str, CANDIDATES)))
OUT = ROOT / 'publication_H1_IPC_ENMIG_ENET_AUC_FULLHEIGHT_240_v2'
OUT.mkdir(parents=True, exist_ok=True)

AGES = ('GW20', 'GW22')
H1 = ('IPC', 'EN-Mig', 'EN-ET')
PAIRS = [('IPC', 'EN-Mig'), ('EN-Mig', 'EN-ET'), ('IPC', 'EN-ET')]
METHODS = ('Zoetrope', 'STORIES')
COLORS = {'Zoetrope': '#D62728', 'STORIES': '#4C78A8'}
DPI = 400

raw = pd.read_csv(SOURCE, dtype={'key': str})
required = {'key', 'age', 'H1', 'Zoetrope_rho_prefix'}
if not required.issubset(raw):
    raise ValueError(f'Missing required fields: {required-set(raw.columns)}')
if raw.key.isna().any() or raw.key.duplicated().any():
    raise RuntimeError('Locked matched-cell CSV contains null or duplicate keys.')
stories_col = next((c for c in ('STORIES_progress_minus_J',
                               'STORIES_maturity_minus_J') if c in raw), None)
if stories_col is None:
    raise ValueError('Could not find original STORIES negative potential (-J).')

cohort = raw.loc[raw.age.isin(AGES) & raw.H1.isin(H1),
                 ['key', 'age', 'H1', 'Zoetrope_rho_prefix', stories_col]].copy()
cohort.rename(columns={'Zoetrope_rho_prefix': 'Zoetrope',
                       stories_col: 'STORIES'}, inplace=True)
for m in METHODS:
    cohort[m] = pd.to_numeric(cohort[m], errors='coerce')
cohort = cohort.loc[np.isfinite(cohort.Zoetrope) & np.isfinite(cohort.STORIES)].copy()
cohort.reset_index(drop=True, inplace=True)
if set(cohort.age.unique()) != set(AGES):
    raise RuntimeError('Both GW20 and GW22 are required in the joint support cohort.')
for age in AGES:
    counts = cohort.loc[cohort.age.eq(age), 'H1'].value_counts()
    if any(counts.get(l, 0) < 20 for l in H1):
        raise RuntimeError(f'Insufficient matched H1 support for {age}: {counts.to_dict()}')
print('Exact matched original cells by H1 and age:')
print(cohort.groupby(['age', 'H1']).size().unstack().reindex(index=AGES, columns=H1))

# ---------------- Within-age percentile ranks ----------------
# The original units of rho and -J are NOT directly comparable.
for age in AGES:
    mask = cohort.age.eq(age).to_numpy()
    for method in METHODS:
        vals = cohort.loc[mask, method].to_numpy(float)
        if np.ptp(vals) < 1e-12:
            raise RuntimeError(f'{age} {method} is constant and has no ranking.')
        cohort.loc[mask, f'{method}_age_percentile'] = (rankdata(vals, method='average')-.5)/len(vals)

# Within each H1 class, both ages have 50% of the statistical weight.
# This ensures the pooled age-adjusted means/boxplots are not affected by the
# GW20:GW22 cell count imbalance in individual H1 classes.
cohort['plot_weight'] = 0.0
for label in H1:
    for age in AGES:
        mask = cohort.H1.eq(label) & cohort.age.eq(age)
        cohort.loc[mask, 'plot_weight'] = 1.0 / (len(AGES) * int(mask.sum()))

# ---------------- AUC: original scores, within-age ----------------
def auc(early, late):
    e = np.asarray(early, float)
    l = np.asarray(late, float)
    if min(len(e), len(l)) < 20:
        return float('nan')
    ranks = rankdata(np.concatenate([e, l]), method='average')
    return float((ranks[len(e):].sum()-len(l)*(len(l)+1)/2)/(len(e)*len(l)))

rows = []
for age in AGES:
    dat = cohort.loc[cohort.age.eq(age)]
    for first, second in PAIRS:
        a = dat.loc[dat.H1.eq(first)]
        b = dat.loc[dat.H1.eq(second)]
        for method in METHODS:
            rows.append(dict(age=age, earlier_H1=first, later_H1=second,
                             method=method, n_early=len(a), n_later=len(b),
                             AUC=auc(a[method], b[method])))
pair_auc = pd.DataFrame(rows)
if not pair_auc.AUC.notna().all():
    raise RuntimeError('An age/H1 pair has undefined AUC.')
age_auc = pair_auc.groupby(['age', 'method'], as_index=False).agg(
    macro_AUC=('AUC', 'mean'), n_pairs=('AUC', 'size'))
pooled_auc = pair_auc.groupby('method', as_index=False).agg(
    age_stratified_macro_AUC=('AUC', 'mean'), n_age_H1_pairs=('AUC', 'size'))
assert (age_auc.n_pairs == len(PAIRS)).all()
assert (pooled_auc.n_age_H1_pairs == len(AGES)*len(PAIRS)).all()
print('\nAge-stratified H1 AUC (native scores, 3 H1 pairs per age):')
print(age_auc.to_string(index=False, float_format=lambda a: f'{a:.4f}'))
print('\nGW20/GW22 equally pooled macro AUC:')
print(pooled_auc.to_string(index=False, float_format=lambda a: f'{a:.4f}'))

# ---------------- Age-balanced box statistics ----------------
def weighted_quantile(v, w, probs):
    """Quantiles of the equal-age-weighted pooled sample (midpoint CDF)."""
    v = np.asarray(v, float)
    w = np.asarray(w, float)
    if not (len(v) and len(v)==len(w) and np.isfinite(v).all() and
            np.isfinite(w).all() and np.all(w>0)):
        raise RuntimeError('Invalid weighted H1 sample')
    order = np.argsort(v, kind='mergesort')
    v = v[order];w = w[order]
    cdf = (np.cumsum(w)-.5*w)/w.sum()
    return np.interp(probs, cdf, v, left=v[0], right=v[-1])

stats = []
records = []
for label in H1:
    sub = cohort.loc[cohort.H1.eq(label)]
    wt = sub.plot_weight.to_numpy(float)
    for method in METHODS:
        vals = sub[f'{method}_age_percentile'].to_numpy(float)
        q1, med, q3 = weighted_quantile(vals, wt, [.25, .5, .75])
        mean = float(np.average(vals, weights=wt))
        iqr = q3-q1
        within = vals[(vals >= q1-1.5*iqr) & (vals <= q3+1.5*iqr)]
        whisklo = float(within.min()) if len(within) else float(q1)
        whiskhi = float(within.max()) if len(within) else float(q3)
        records.append(dict(H1=label, method=method, mean_age_adjusted_percentile=mean,
                            q1=q1, median=med, q3=q3, whislo=whisklo, whishi=whiskhi,
                            GW20_n=int((sub.age=='GW20').sum()),
                            GW22_n=int((sub.age=='GW22').sum())))
        stats.append(dict(label=label, method=method, mean=mean,
                          med=med, q1=q1, q3=q3, whislo=whisklo,
                          whishi=whiskhi, fliers=[]))
box_table = pd.DataFrame(records)

# ---------------- Publication figure | MATCHED-HEIGHT BOX + AUC ----------------
# Fixed physical dimensions inherited from the previous compact figure.
# Separate original-score AUC from the display's within-age percentiles.
# AUC y-axis starts at ZERO; 0.5 reference is unskewed chance performance.
INK = '#26343D'
GRID = '#CFD5DB'
mpl.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 8.2,
    'axes.labelsize': 8.0, 'axes.titlesize': 9.1, 'axes.titleweight': 'bold',
    'xtick.labelsize': 8.0, 'ytick.labelsize': 7.6, 'axes.linewidth': .72,
    'axes.edgecolor': INK, 'axes.labelcolor': INK, 'text.color': INK,
    'xtick.color': INK, 'ytick.color': INK,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'savefig.facecolor': 'white', 'figure.facecolor': 'white',
})
FIG_W, FIG_H = 2.40, 2.55
DPI = 450
fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor='white')

# Both axes have IDENTICAL bottom and height. A narrower main plot frees
# a deliberate 0.396-inch gutter for the right panel's vertical AUC label.
BOTTOM, HEIGHT = .325, .530
BOX_POS = [.195, BOTTOM, .395, HEIGHT]
AUC_POS = [.755, BOTTOM, .225, HEIGHT]
ax = fig.add_axes(BOX_POS)
auc_ax = fig.add_axes(AUC_POS)
assert abs(ax.get_position().height - auc_ax.get_position().height) < 1e-12

centers = np.arange(len(H1), dtype=float)
for method, shift in [('Zoetrope', -.145), ('STORIES', .145)]:
    subset = [next(s for s in stats if s['label'] == label and s['method'] == method)
              for label in H1]
    ax.bxp(subset, positions=centers + shift, widths=.235,
           patch_artist=True, showfliers=False, showmeans=True,
           manage_ticks=False,
           boxprops=dict(facecolor=COLORS[method], edgecolor=COLORS[method],
                         alpha=.85, linewidth=.75),
           medianprops=dict(color='white', linewidth=.85),
           whiskerprops=dict(color=COLORS[method], linewidth=.72),
           capprops=dict(color=COLORS[method], linewidth=.72),
           meanprops=dict(marker='D', markersize=2.9,
                          markerfacecolor=INK, markeredgecolor='white',
                          markeredgewidth=.35, linestyle='none'))
    for x, item in zip(centers, subset):
        # Distinct row per model: these are equal-age-weighted means.
        y = 1.106 if method == 'Zoetrope' else 1.037
        ax.text(x, y, f'{item["mean"]:.2f}', color=COLORS[method],
                ha='center', va='center', fontsize=7.0, fontweight='bold',
                clip_on=False)

ax.set_xlim(-.46, len(H1) - .54)
ax.set_ylim(0, 1.175)  # displayed means are above 0..1 percentile support
ax.set_xticks(centers, ['IPC', 'EN-\nMig', 'EN-\nET'])
ax.set_yticks([0, .5, 1.], ['0', '0.5', '1'])
ax.set_ylabel('Age-adjusted percentile', fontsize=8.0, labelpad=1.7)
ax.grid(axis='y', color=GRID, lw=.55, alpha=.60)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(axis='x', length=0, pad=2)
ax.tick_params(axis='y', length=2.2, width=.65, pad=2)

# Skinny fully scaled AUC panel, precisely as tall as the boxplot panel.
auc_values = pooled_auc.set_index('method')['age_stratified_macro_AUC'].astype(float)
bar_x = np.arange(len(METHODS), dtype=float)
bars = auc_ax.bar(bar_x, [float(auc_values[m]) for m in METHODS],
                  width=.68, color=[COLORS[m] for m in METHODS],
                  edgecolor='none', zorder=3)
auc_ax.set_ylim(0, 1.0)
auc_ax.set_xlim(-.52, 1.52)
auc_ax.set_yticks([0, .5, 1.], ['0', '0.5', '1'])
auc_ax.set_ylabel('AUC', fontsize=8.8, fontweight='bold', labelpad=1.0)
# Method names are in the shared legend, preventing crowded 0.54-inch x ticks.
auc_ax.set_xticks(bar_x, ['Z', 'S'])
auc_ax.axhline(.5, color='#75818D', linestyle=(0,(2.5,2)), linewidth=.85,
               zorder=2, alpha=.85)
auc_ax.grid(axis='y', color=GRID, lw=.55, alpha=.60)
auc_ax.set_axisbelow(True)
auc_ax.spines[['top', 'right']].set_visible(False)
auc_ax.tick_params(axis='x', length=0, pad=2.0, labelsize=7.8)
auc_ax.tick_params(axis='y', length=2.2, width=.65, pad=1.5, labelsize=7.0)
for bar, method in zip(bars, METHODS):
    val = float(auc_values[method])
    # High values remain safely below the top of the fixed 0-1 axes.
    auc_ax.text(bar.get_x() + bar.get_width()/2, min(val+.018, .99),
                f'{val:.3f}', ha='center', va='bottom',
                color=COLORS[method], fontsize=6.8, fontweight='bold',
                clip_on=False)

legend = [
    Patch(facecolor=COLORS['Zoetrope'], edgecolor='none', label='Zoetrope (ρ)'),
    Patch(facecolor=COLORS['STORIES'], edgecolor='none', label='STORIES (−J)'),
    Line2D([0], [0], marker='D', linestyle='none', markersize=3.8,
           markerfacecolor=INK, markeredgecolor='white', label='Mean'),
]
fig.legend(handles=legend, ncol=3, loc='lower center',
           bbox_to_anchor=(.50, .105), frameon=False, fontsize=6.6,
           columnspacing=.60, handlelength=.78, handletextpad=.28,
           borderaxespad=0.)

# Generate all outputs FIRST: optional font checks should never prevent saving.
STEM = 'FIG_Zoetrope_STORIES_IPC_ENMIG_ENET_GW20_GW22_MATCHED_HEIGHT_AUC_240'
for suffix in ('png', 'pdf', 'svg'):
    target = OUT / f'{STEM}.{suffix}'
    fig.savefig(target, dpi=DPI if suffix == 'png' else None,
                facecolor='white', bbox_inches=None, pad_inches=0)
    if not target.is_file() or target.stat().st_size < 500:
        raise RuntimeError(f'Failed to save publication figure: {target}')
    print('SAVED', target)

# Non-fatal geometry checks for other font environments.
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
for item in list(ax.texts) + list(auc_ax.texts) + [ax.yaxis.label, auc_ax.yaxis.label]:
    b = item.get_window_extent(renderer).transformed(fig.dpi_scale_trans.inverted())
    if b.x0 < -.015 or b.x1 > FIG_W+.015 or b.y0 < -.015 or b.y1 > FIG_H+.015:
        print('LAYOUT WARNING:', repr(item.get_text()), tuple(round(float(v),4) for v in b.bounds))

plt.show()
plt.close(fig)

# ---------------- Provenance / tables ----------------
cohort.to_csv(OUT/'JOINT_SUPPORT_H1_AGE_ADJUSTED_CELL_SCORES.csv',index=False)
box_table.to_csv(OUT/'POOLED_H1_AGE_BALANCED_MEANS_AND_BOXPLOT_STATS.csv',index=False)
pair_auc.to_csv(OUT/'NATIVE_H1_ORDERING_AUC_BY_AGE_PAIR.csv',index=False)
age_auc.to_csv(OUT/'NATIVE_H1_ORDERING_MACRO_AUC_BY_AGE.csv',index=False)
pooled_auc.to_csv(OUT/'NATIVE_H1_ORDERING_EQUAL_AGE_MACRO_AUC.csv',index=False)
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as fp:
        for block in iter(lambda:fp.read(4*1024*1024),b''):
            h.update(block)
    return h.hexdigest()
receipt = dict(source_file=str(SOURCE),source_sha256=sha256(SOURCE),
               ages=list(AGES),H1_order=list(H1),n_joint_cells=len(cohort),
               score_map={'Zoetrope':'Zoetrope_rho_prefix', 'STORIES':stories_col},
               display='Within-age percentile ranks pooled across GW20 and GW22',
               box_and_mean_age_weighting='Each GW contributes equally within each H1 and method',
               mean_definition='Equal-weight mean of GW20 and GW22 within-age percentile ranks for each H1',
               box_whisker_rule='Weighted Q1/median/Q3, whiskers to observations within 1.5 IQR',
               auc_definition='3 ordered H1 pairs x 2 ages; equal-weight mean of within-age native-score pairwise AUC',
               AUC_not_from_pooled_cells=True,
               GW15_excluded='Zoetrope rho=0 by construction at GW15',
               exploratory_in_sample=True,
               training_data_asymmetry='Zoetrope trained on 9 sections; STORIES on 3',
               figure_size_inches=[FIG_W,FIG_H],
               auc_y_axis=[0,1], auc_y_axis_label='AUC', auc_matches_boxplot_height=True,
               auc_pairs=[list(x) for x in PAIRS],
               method_colors=COLORS)
(OUT/'FIGURE_RECEIPT.json').write_text(json.dumps(receipt,indent=2)+'\n')
print('\nSaved figure (PNG/PDF/SVG) and tables to:', OUT)
print('\nAge-adjusted, age-balanced mean percentiles by H1:')
print(box_table.pivot(index='H1',columns='method',values='mean_age_adjusted_percentile')
      .reindex(index=H1,columns=METHODS).round(3))

## Optional replay of STORIES scores for the frozen cohort

Run this section by itself after setup before the score cell only if rebuilding the matched score table from the frozen cell manifest and the STORIES checkpoint. Set ZOETROPE_EXPORT_PFC_PROGRESSION=1 before running this cell.

In [ ]:
if os.environ.get("ZOETROPE_EXPORT_PFC_PROGRESSION", "0") == "1":

    # ============================================================
    # ZOETROPE vs STORIES
    # H1 DEVELOPMENTAL ORDERING — DEFAULT PYTHON ENVIRONMENT
    #
    # No stories / flax / jax imports
    # No cell transport or retraining
    #
    # Requires previous Zoetrope export:
    #   PART_A_LOCKED_MATCHED_CELLS.csv
    #
    # Output:
    #   COPY_PASTE_SCALAR_ORDERING.txt
    #   PAIRWISE_H1_ORDERING_AUC.csv
    #   MACRO_AUC_BY_AGE.csv
    #   OVERALL_MACRO_AUC.csv
    # ============================================================

    import os
    import json
    import hashlib
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import h5py
    import msgpack
    from scipy.stats import rankdata

    # ------------------------------------------------------------
    # 1. Paths
    # ------------------------------------------------------------

    P = Path(os.environ.get(
        "SPATIAL_TI_PROJECT",
        os.environ["STP_PROJECT"]
    ))

    ROOT = Path(os.environ.get(
        "QIAN_ORDERING_BENCH_OUT",
        str(P / "paper_figures/qian_PFC_Zoetrope_vs_STORIES_ordering_v1")
    ))

    STO = Path(os.environ.get(
        "QIAN_STORIES_FROZEN_DIR",
        str(
            P / "experiments/qian_pfc_publication_4way_20261024"
                "/stories_slices_1_2_7_tutorial_GPU"
        )
    ))

    OUT = ROOT / "PART_A_POTENTIAL_ONLY_DEFAULT_ENV"
    OUT.mkdir(parents=True, exist_ok=True)

    required_files = [
        ROOT / "PART_A_LOCKED_MATCHED_CELLS.csv",
        STO / "SUCCESS.json",
        STO / "run_config.json",
        STO / "native_best_params.msgpack",
        STO / "preprocessing_frozen.npz",
        STO / "stories_input_3tissue.h5ad",
        STO / "native_potential_QC.npz",
    ]

    for path in required_files:
        if not path.is_file():
            raise FileNotFoundError(
                f"Missing required artifact: {path}\n"
                "Check the STORIES path or previous Zoetrope export."
            )

    # ------------------------------------------------------------
    # 2. Verify frozen STORIES model
    # ------------------------------------------------------------

    def sha256(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for block in iter(
                lambda: f.read(4 * 1024 * 1024), b""
            ):
                h.update(block)
        return h.hexdigest()


    success = json.loads((STO / "SUCCESS.json").read_text())
    config = json.loads((STO / "run_config.json").read_text())

    with np.load(
        STO / "preprocessing_frozen.npz",
        allow_pickle=False
    ) as f:
        sections = f["section_ids"].astype(int).tolist()
        age_times = f["times"].astype(float)
        genes = f["genes"].astype(str).tolist()

    assert (
        success.get("status") == "success"
        and success.get("smoke") is False
    ), "Not a successful full STORIES fit"

    assert (
        config.get("method") == "STORIES"
        and config.get("smoke") is False
    ), "Wrong STORIES configuration"

    assert (
        config.get("training_sections") == sections == [1, 2, 7]
    ), "Incorrect training sections"

    assert np.allclose(
        age_times, [0, 5 / 7, 1]
    ), "Incorrect training ages"

    assert (
        success.get("n_cells")
        == config.get("dataset_cells")
        == 1_515_682
    ), "Full training-cell count mismatch"

    assert (
        success.get("n_genes")
        == config.get("genes")
        == len(genes)
        == 300
    ), "Gene-panel mismatch"

    assert (
        config.get("fit", {}).get("max_iter") == 10_000
        and config.get("fit", {}).get("batch_size") == 1_000
    ), "Not the full STORIES training configuration"

    for filename, field in [
        ("native_best_params.msgpack", "model_sha256"),
        ("preprocessing_frozen.npz", "preprocessing_sha256"),
    ]:
        assert (
            sha256(STO / filename) == success[field]
        ), f"STORIES artifact checksum mismatch: {filename}"

    print(
        "PASS frozen full STORIES model, correct sections, "
        "original training settings and hashes."
    )

    # ------------------------------------------------------------
    # 3. Decode saved Flax weights using NumPy
    # ------------------------------------------------------------

    def unpack_ext(code, data):
        if code in (1, 3):
            shape, dtype, raw = msgpack.unpackb(
                data, raw=False
            )

            if dtype == "bfloat16":
                raise RuntimeError(
                    "Unexpected bfloat16 checkpoint weights"
                )

            arr = np.frombuffer(
                raw, dtype=np.dtype(dtype)
            ).reshape(tuple(shape))

            return arr[()] if code == 3 else arr

        if code == 2:
            real, imag = msgpack.unpackb(
                data, raw=False
            )
            return complex(real, imag)

        raise RuntimeError(
            f"Unexpected Flax msgpack extension: {code}"
        )


    state = msgpack.unpackb(
        (STO / "native_best_params.msgpack").read_bytes(),
        ext_hook=unpack_ext,
        raw=False,
        strict_map_key=False,
    )

    params = state.get("params", state)

    assert isinstance(params, dict), (
        f"Unexpected checkpoint structure: {list(state)}"
    )

    layers = []

    for i, dims in enumerate([
        (20, 128),
        (128, 128),
        (128, 1),
    ]):
        key = f"Dense_{i}"

        assert key in params, (
            f"Missing {key}: {list(params)}"
        )

        W = np.asarray(
            params[key]["kernel"],
            dtype=np.float32,
        )

        assert W.shape == dims, (
            key, W.shape, dims
        )

        if i < 2:
            b = np.asarray(
                params[key]["bias"],
                dtype=np.float32,
            )
            assert b.shape == (dims[1],)
        else:
            assert "bias" not in params[key]
            b = None

        layers.append((W, b))


    def stories_J_numpy(X, batch=4096):
        """
        Evaluate the frozen STORIES differentiation potential.

        Uses the original saved weights and the GeLU
        activation, without importing STORIES or JAX.
        """
        X = np.asarray(X, dtype=np.float32)

        assert X.ndim == 2 and X.shape[1] == 20

        ans = np.empty(len(X), dtype=np.float32)

        for i in range(0, len(X), batch):
            h = X[i:i + batch]

            for W, b in layers[:2]:
                h = h @ W + b

                h = h * (
                    np.float32(0.5)
                    * (
                        1 + np.tanh(
                            np.float32(np.sqrt(2 / np.pi))
                            * (
                                h
                                + np.float32(0.044715) * h**3
                            )
                        )
                    )
                )

            ans[i:i + batch] = (
                h @ layers[2][0]
            ).reshape(-1)

        if not np.isfinite(ans).all():
            raise RuntimeError(
                "STORIES potential returned nonfinite values"
            )

        return ans.astype(float)

    # ------------------------------------------------------------
    # 4. Load matched cell cohort
    # ------------------------------------------------------------

    cohort = pd.read_csv(
        ROOT / "PART_A_LOCKED_MATCHED_CELLS.csv",
        dtype={"key": str},
    )

    required_columns = {
        "key",
        "sid",
        "age",
        "H1",
        "z",
        "Zoetrope_z_anatomical",
        "Zoetrope_rho_prefix",
    }

    assert required_columns.issubset(cohort.columns), (
        "Locked cohort missing columns: "
        f"{required_columns - set(cohort.columns)}"
    )

    assert cohort.key.is_unique
    assert set(cohort.sid.astype(int)) == {1, 2, 7}

    # ------------------------------------------------------------
    # 5. Load native STORIES potential QC reference
    # ------------------------------------------------------------

    with np.load(
        STO / "native_potential_QC.npz",
        allow_pickle=False,
    ) as f:
        qc_keys = f["cell_key"].astype(str)
        qc_potential = f["potential"].astype(float)

    assert (
        len(qc_keys) == len(qc_potential)
        and len(qc_keys) > 500
    )

    # ------------------------------------------------------------
    # 6. Read original STORIES embedding
    #
    # FIX: AnnData may store its named observation index
    # under "cell_key" rather than "_index".
    # ------------------------------------------------------------

    with h5py.File(
        STO / "stories_input_3tissue.h5ad", "r"
    ) as f:

        obs_group = f["obs"]

        index_name = obs_group.attrs.get(
            "_index", "cell_key"
        )

        if isinstance(index_name, bytes):
            index_name = index_name.decode("utf-8")

        if index_name not in obs_group:
            raise KeyError(
                "Cannot find observation index dataset "
                f"{index_name!r}.\n"
                "Available datasets: "
                f"{list(obs_group.keys())[:35]}"
            )

        ds = obs_group[index_name]

        full_keys = pd.Index(
            ds.asstr()[:]
        )

        assert len(full_keys) == 1_515_682
        assert full_keys.is_unique

        # Exact cell-ID matching; never assume row order.
        idx = full_keys.get_indexer(
            cohort.key.to_numpy(str)
        )

        qc_idx = full_keys.get_indexer(qc_keys)

        if (idx < 0).any() or (qc_idx < 0).any():
            raise RuntimeError(
                "Strict STORIES cell-ID matching failed. "
                f"Cohort missing: {(idx < 0).sum()}, "
                f"QC missing: {(qc_idx < 0).sum()}."
            )

        embedding_dataset = f["obsm"]["X_pca_harmony"]

        assert embedding_dataset.shape == (
            1_515_682, 20
        ), embedding_dataset.shape

        # Read only matched and QC rows to save memory.
        joined = np.unique(
            np.r_[idx, qc_idx]
        )

        embedding = np.asarray(
            embedding_dataset[joined, :],
            dtype=np.float32,
        )

        positions = pd.Index(joined)

        X = embedding[
            positions.get_indexer(idx)
        ]

        qc_X = embedding[
            positions.get_indexer(qc_idx)
        ]

        # Verify matched cells belong to the correct tissue.
        sids = np.asarray(
            obs_group["original_slice_id"][:],
            dtype=int,
        )

        assert np.array_equal(
            sids[idx],
            cohort.sid.to_numpy(int),
        ), "Matched tissue identities disagree"

    # ------------------------------------------------------------
    # 7. Mandatory native-potential verification
    # ------------------------------------------------------------

    qc_replayed = stories_J_numpy(qc_X)

    errors = np.abs(
        qc_replayed - qc_potential
    )

    mae = float(np.mean(errors))
    max_error = float(np.max(errors))
    scale = float(np.max(np.abs(qc_potential)))

    print(
        "Native STORIES potential QC:",
        f"n={len(qc_keys):,}",
        f"MAE={mae:.6g}",
        f"max_abs_error={max_error:.6g}",
    )

    if max_error > 2e-3 + 1e-4 * scale:
        raise RuntimeError(
            "NumPy potential replay does not match native "
            "STORIES potentials. Benchmark stopped."
        )

    print(
        "PASS NumPy replay matches native STORIES "
        "potentials; no STORIES import or transport."
    )

    # ------------------------------------------------------------
    # 8. Compute observed-cell differentiation scores
    # ------------------------------------------------------------

    cohort["STORIES_native_potential_J"] = (
        stories_J_numpy(X)
    )

    # High STORIES potential = lower differentiation.
    cohort["STORIES_progress_minus_J"] = (
        -cohort["STORIES_native_potential_J"]
    )

    # ------------------------------------------------------------
    # 9. Biological differentiation hierarchy
    #
    # EN-IT and EN-ET are separate terminal branches.
    # Do not impose an ordering between them.
    # ------------------------------------------------------------

    ORDER = [
        ("RG", "IPC"),
        ("IPC", "EN-Mig"),
        ("EN-Mig", "EN-IT"),
        ("EN-Mig", "EN-ET"),
        ("RG", "EN-Mig"),
        ("RG", "EN-IT"),
        ("RG", "EN-ET"),
        ("IPC", "EN-IT"),
        ("IPC", "EN-ET"),
    ]

    FIELDS = [
        "STORIES_progress_minus_J",
        "Zoetrope_z_anatomical",
        "Zoetrope_rho_prefix",
    ]

    # ------------------------------------------------------------
    # 10. Pairwise developmental-ordering AUC
    # ------------------------------------------------------------

    def auc(early, later):
        """
        Probability that a later-stage biological class
        receives a higher progression score than an
        earlier class.

        Ties receive half credit.
        """
        a = np.asarray(early, dtype=float)
        b = np.asarray(later, dtype=float)

        a = a[np.isfinite(a)]
        b = b[np.isfinite(b)]

        if min(len(a), len(b)) < 20:
            return np.nan, len(a), len(b)

        values = np.r_[a, b]

        if np.ptp(values) < 1e-10:
            return np.nan, len(a), len(b)

        ranks = rankdata(
            values,
            method="average",
        )

        n_early = len(a)
        n_later = len(b)

        U = (
            ranks[n_early:].sum()
            - n_later * (n_later + 1) / 2
        )

        score = U / (n_early * n_later)

        return float(score), n_early, n_later


    rows = []

    for age, sub in cohort.groupby(
        "age", sort=True
    ):
        for early, later in ORDER:

            a = sub.loc[
                sub.H1.eq(early)
            ]

            b = sub.loc[
                sub.H1.eq(later)
            ]

            for field in FIELDS:

                score, ne, nl = auc(
                    a[field],
                    b[field],
                )

                rows.append({
                    "age": age,
                    "early_H1": early,
                    "later_H1": later,
                    "method": field,
                    "n_early": ne,
                    "n_later": nl,
                    "ordering_AUC": score,
                })

    pairs = pd.DataFrame(rows)

    # ------------------------------------------------------------
    # 11. Summary statistics
    # ------------------------------------------------------------

    macro = (
        pairs.groupby(
            ["age", "method"],
            as_index=False,
        )
        .agg(
            macro_AUC=("ordering_AUC", "mean"),
            valid_pairs=("ordering_AUC", "count"),
        )
    )

    coverage = (
        cohort.groupby(["age", "H1"])
        .agg(
            n=("key", "size"),
            rho_supported=(
                "Zoetrope_rho_prefix",
                lambda x: int(
                    np.isfinite(x).sum()
                ),
            ),
        )
        .reset_index()
    )

    pooled = (
        pairs.groupby(
            "method",
            as_index=False,
        )
        .agg(
            across_age_pair_macro_AUC=(
                "ordering_AUC", "mean"
            ),
            valid_age_pairs=(
                "ordering_AUC", "count"
            ),
        )
    )

    # ------------------------------------------------------------
    # 12. Save complete results
    # ------------------------------------------------------------

    cohort.to_csv(
        OUT / "MATCHED_CELLS_STORIES_NATIVE_POTENTIAL.csv",
        index=False,
    )

    pairs.to_csv(
        OUT / "PAIRWISE_H1_ORDERING_AUC.csv",
        index=False,
    )

    macro.to_csv(
        OUT / "MACRO_AUC_BY_AGE.csv",
        index=False,
    )

    coverage.to_csv(
        OUT / "MATCHED_H1_COVERAGE.csv",
        index=False,
    )

    pooled.to_csv(
        OUT / "OVERALL_MACRO_AUC.csv",
        index=False,
    )

    # ------------------------------------------------------------
    # 13. Print copy-pasteable results
    # ------------------------------------------------------------

    log = []

    def report(s=""):
        log.append(str(s))
        print(s, flush=True)


    report(
        "ZOETROPE vs STORIES | "
        "SCALAR DIFFERENTIATION ORDERING | DEFAULT ENV"
    )

    report(
        "Frozen STORIES potential replayed using NumPy; "
        f"native {len(qc_keys):,}-cell QC passed."
    )

    report(
        "Full non-smoke STORIES fit: sections [1,2,7]. "
        f"Matched original cells: {len(cohort):,}."
    )

    report(
        "Biological hierarchy: RG -> IPC -> EN-Mig -> "
        "{EN-IT, EN-ET}."
    )

    report(
        "AUC = Pr(score(later) > score(earlier)), "
        "ties count half; 0.5 chance, 1.0 perfect."
    )

    report(
        "STORIES score = -J. Zoetrope z is anatomical; "
        "rho is Fisher arclength, NOT differentiation."
    )

    report(
        "GW15 rho is zero by definition; within-age AUC "
        "is undefined. Unsupported rho remains missing."
    )

    report(
        "All ages participated in fitting: this is "
        "IN-SAMPLE, not a held-out prediction benchmark."
    )

    report(
        f"Native potential QC: n={len(qc_keys)}, "
        f"mean_abs_err={mae:.6g}, "
        f"max_abs_err={max_error:.6g}"
    )

    report("\n--- MACRO AUC BY AGE ---")
    report(
        macro.round(4).to_string(
            index=False
        )
    )

    report(
        "\n--- ACROSS-AGE MACRO OF WITHIN-AGE PAIRS ---"
    )
    report(
        pooled.round(4).to_string(
            index=False
        )
    )

    report("\n--- INDIVIDUAL BIOLOGICAL PAIRS ---")
    report(
        pairs.round(4).to_string(
            index=False
        )
    )

    report("\n--- CLASS COVERAGE / SUPPORTED RHO ---")
    report(
        coverage.to_string(
            index=False
        )
    )

    report("\nSAVED: " + str(OUT))

    (
        OUT / "COPY_PASTE_SCALAR_ORDERING.txt"
    ).write_text(
        "\n".join(log) + "\n"
    )

    print(
        "\nFINISHED. Paste COPY_PASTE_SCALAR_ORDERING.txt "
        "here for interpretation."
    )